In [1]:


import pandas as pd
import random

random.seed(42)

# Define data parameters
regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Books']
salespersons = ['Alice', 'Bob', 'Carol', 'David', 'Emma', 'Frank']

# Generate 200 sales transactions
data = {
    'transaction_id': range(1001, 1201),
    'region': [random.choice(regions) for _ in range(200)],
    'category': [random.choice(categories) for _ in range(200)],
    'salesperson': [random.choice(salespersons) for _ in range(200)],
    'sales_amount': [round(random.uniform(50, 5000), 2) for _ in range(200)],
    'customer_id': [random.randint(5000, 5100) for _ in range(200)]
}

df = pd.DataFrame(data)
print(df.head(10))
print(f"\nDataset shape: {df.shape}")


   transaction_id region     category salesperson  sales_amount  customer_id
0            1001  North  Electronics        Emma       4000.80         5075
1            1002  North        Books       Alice       3634.70         5084
2            1003   East  Electronics       David       4210.50         5025
3            1004  South  Electronics        Emma       4601.73         5054
4            1005  South     Clothing         Bob       4904.57         5014
5            1006  South     Clothing       Carol       2693.91         5069
6            1007  North       Sports       Alice       4539.30         5028
7            1008  North       Sports       Frank       2979.87         5082
8            1009   West       Sports       David       3331.85         5019
9            1010  North     Clothing       Alice        465.54         5034

Dataset shape: (200, 6)


In [21]:
#Task 1: Basic Grouping and Single Aggregations


#Calculate total sales amount for each region using groupby() and sum()
region_sales = df.groupby('region')['sales_amount'].sum().reset_index()
print("===Total Region Sales===\n",region_sales)

#Count the number of transactions for each product category using groupby() and count()
category_transactions = df.groupby('category')['transaction_id'].count().reset_index()
print("\n===Number of Transations===\n",category_transactions)

#Calculate the average sales amount per salesperson using groupby() and mean()
avg_sales = df.groupby('salesperson')['sales_amount'].mean().reset_index()
print("\n===Average Sales===\n",avg_sales)

#Sort the regional sales results in descending order using sort_values(ascending=False) to identify the top-performing region
region_sales = df.groupby('region')['sales_amount'].sum().sort_values(ascending=False).reset_index()

print("\n===Sorted Region Sales===\n",region_sales)


===Total Region Sales===
   region  sales_amount
0   East      95189.81
1  North     135181.16
2  South     158977.36
3   West     109383.07

===Number of Transations===
         category  transaction_id
0          Books              39
1       Clothing              42
2    Electronics              45
3  Home & Garden              43
4         Sports              31

===Average Sales===
   salesperson  sales_amount
0       Alice   1998.432273
1         Bob   2554.919063
2       Carol   2454.368571
3       David   2743.036444
4        Emma   2493.457273
5       Frank   2464.135750

===Sorted Region Sales===
   region  sales_amount
0  South     158977.36
1  North     135181.16
2   West     109383.07
3   East      95189.81


In [34]:
#Task 2: Multi-Column Grouping and Multiple Aggregations
#Group by both region AND category to calculate total sales for each combination using:
total_sales_region_category = df.groupby(['region', 'category'])['sales_amount'].sum().reset_index()
print(total_sales_region_category)

#For each salesperson, calculate three metrics simultaneously
df.groupby('salesperson')['sales_amount'].agg(['sum', 'mean', 'count']).reset_index()

#Sort the salesperson results by total sales in descending order to identify the top performer
sales_summary = df.groupby('salesperson')['sales_amount'] \
                 .agg(total_sales='sum', average_sales='mean', transactions='count') \
                  .reset_index()

print("\n===Sales Summary===\n", sales_summary)

sales_summary_sorted = sales_summary.sort_values(by='total_sales', ascending=False)

print("\n===Top Performer===\n",sales_summary_sorted)


#Use .idxmax() on the grouped category sales to find which category has the maximum total revenue
top_category = total_sales_region_category .idxmax()

print("\n===Top revenue category===\n", top_category)

   region       category  sales_amount
0    East          Books      20027.53
1    East       Clothing      19926.20
2    East    Electronics      22791.55
3    East  Home & Garden      15949.08
4    East         Sports      16495.45
5   North          Books      42592.53
6   North       Clothing      18959.06
7   North    Electronics      38889.84
8   North  Home & Garden      19344.48
9   North         Sports      15395.25
10  South          Books      21007.68
11  South       Clothing      50878.36
12  South    Electronics      39229.63
13  South  Home & Garden      22592.00
14  South         Sports      25269.69
15   West          Books      20359.41
16   West       Clothing      16548.81
17   West    Electronics      13553.27
18   West  Home & Garden      33069.36
19   West         Sports      25852.22

===Sales Summary===
   salesperson  total_sales  average_sales  transactions
0       Alice     43965.51    1998.432273            22
1         Bob     81757.41    2554.919063      

In [46]:
#Task 3: Custom Aggregation and Complete Sales Report

#Define a custom aggregation function that calculates the sales range (max - min) for each group:
sales_range_per_person = df.groupby('salesperson')['sales_amount'].agg(sales_range).reset_index()
print(sales_range_per_person)


#Apply this custom function along with standard aggregations to analyze sales by region:

region_sales_summary = df.groupby('region')['sales_amount'].agg(
    ['sum', 'mean', 'min', 'max', sales_range]
).reset_index()

#Create a final summary report
region_summary = df.groupby('region').agg(
    total_transactions=('customer_id', 'count'),   # number of transactions
    total_sales=('sales_amount', 'sum'),           # total sales amount
    average_transaction=('sales_amount', 'mean')   # average transaction value
).reset_index()

print("\n===Final Summary Report===\n",region_sales_summary)


#Explain the multi-level column structure
print("\nWhen you do dictionary-based aggregation in pandas, the resulting DataFrame can have a MultiIndex for its columns, whereas a single aggregation produces a flat column structure.\n")
print("\nThe columns have a MultiIndex:\n Level 0: original column names → sales_amount, customer_id\nLevel 1: aggregation functions → sum, mean, count")

  salesperson  sales_amount
0       Alice       4403.95
1         Bob       4791.94
2       Carol       4691.83
3       David       4545.19
4        Emma       4734.05
5       Frank       4650.24

===Final Summary Report===
   region        sum         mean     min      max  sales_range
0   East   95189.81  2213.716512  307.82  4758.46      4450.64
1  North  135181.16  2413.949286  110.71  4788.56      4677.85
2  South  158977.36  2789.076491  373.45  4983.33      4609.88
3   West  109383.07  2485.978864  203.60  4903.53      4699.93

When you do dictionary-based aggregation in pandas, the resulting DataFrame can have a MultiIndex for its columns, whereas a single aggregation produces a flat column structure.


The columns have a MultiIndex:
 Level 0: original column names → sales_amount, customer_id
Level 1: aggregation functions → sum, mean, count
